# 🏬 Topic 09: Feature Stores & ML Governance

## 1. Why Feature Stores?
In organizations with multiple ML models, feature engineering logic is often duplicated, leading to:
1. **Training-Serving Skew:** Different code used to calculate features for training vs online inference.
2. **Point-In-Time Data Leakage:** Joining historical features without respecting event timestamps.

### Dual Storage Engine Pattern:
- **Offline Store (Batch):** Data Lake / Parquet / S3 for large-scale training dataset generation.
- **Online Store (Real-time):** Redis / DynamoDB for sub-10ms feature lookup during serving.

---

## 2. Hands-on: Building a Point-in-Time Correct Feature Store


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

class MiniFeatureStore:
    def __init__(self):
        self.feature_table = pd.DataFrame()

    def ingest_features(self, df: pd.DataFrame):
        self.feature_table = pd.concat([self.feature_table, df], ignore_index=True)
        print(f"📥 Ingested {len(df)} feature records into Feature Store.")

    def get_historical_features(self, entity_df: pd.DataFrame):
        results = []
        for idx, row in entity_df.iterrows():
            user_id = row['user_id']
            query_time = row['timestamp']
            
            matching = self.feature_table[
                (self.feature_table['user_id'] == user_id) & 
                (self.feature_table['timestamp'] <= query_time)
            ]
            
            if not matching.empty:
                latest_feature = matching.sort_values('timestamp').iloc[-1].to_dict()
                results.append(latest_feature)
                
        return pd.DataFrame(results)

store = MiniFeatureStore()

store.ingest_features(pd.DataFrame([
    {"user_id": 101, "timestamp": datetime(2026, 1, 1), "credit_score": 650, "avg_spend": 120.0},
    {"user_id": 101, "timestamp": datetime(2026, 6, 1), "credit_score": 720, "avg_spend": 250.0},
]))

observation_df = pd.DataFrame([
    {"user_id": 101, "timestamp": datetime(2026, 3, 15)}
])

res = store.get_historical_features(observation_df)
print("\n🔍 Point-In-Time Training Feature Match:")
print(res[["user_id", "timestamp", "credit_score", "avg_spend"]])
